# Import & Setup

In [3]:
import os
if os.path.exists('digit_recognizer'):
    !rm -rf digit_recognizer
!git clone https://github.com/Sirius-Siru/digit_recognizer.git

import sys
if '/content/digit_recognizer' not in sys.path:
    sys.path.append('/content/digit_recognizer')

!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

if os.path.isdir('data') and os.path.exists('data/train.csv'):
    print("✨ Dữ liệu đã sẵn sàng, không cần tải lại.")
else:
    print("🚀 Đang tải dữ liệu...")
    !kaggle competitions download -c digit-recognizer
    !unzip -o digit-recognizer.zip -d data
    !rm -f digit-recognizer.zip

Cloning into 'digit_recognizer'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 153 (delta 81), reused 120 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 478.75 KiB | 3.39 MiB/s, done.
Resolving deltas: 100% (81/81), done.
🚀 Đang tải dữ liệu...
100% 15.3M/15.3M [00:01<00:00, 10.3MB/s]

Archive:  digit-recognizer.zip
  inflating: data/sample_submission.csv  
  inflating: data/test.csv           
  inflating: data/train.csv          


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import gc

# Load data

In [5]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

X = train.drop('label', axis=1)
y = train['label']

# Unflatten & Normalize
X = X.values.reshape(-1, 28, 28, 1) / 255.0
test = test.values.reshape(-1, 28, 28, 1) / 255.0

# Setup model

In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    # Lớp Conv đầu tiên: Tìm các nét thẳng, cong (như cái bụng số 9)
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Lớp Conv thứ hai: Kết hợp các nét thành hình dạng phức tạp hơn
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Phẳng hóa để đưa vào lớp phân loại cuối cùng
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax') # 10 đầu ra tương ứng 10 chữ số
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [9]:
# Train-Test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Train model

In [ ]:
print(y[len(y) * 80 // 100 :].unique().sum())

45


In [ ]:
model.fit(X, y, epochs=5, batch_size=64, validation_split=0.2)